# Tester

> **Reference:** Barrett et al., *arXiv:2502.12870* (2025). GitHub: https://github.com/rhyan10/X-MACE

## 1. Dataset

In [5]:
import ase.io
import numpy as np

train_file = "data/A01_ethene_grid_static_CASSCF.xyz"
db = ase.io.read(train_file, ":")

atoms = db[0]

# Automatically infer dataset dimensions
n_atoms = len(atoms)
n_geometries = len(db)
n_energies = np.asarray(atoms.info["REF_energy"]).shape[1]

print(f"Atoms per frame      : {n_atoms}")
print(f"Electronic states    : {n_energies}")
print(f"Number of geometries : {n_geometries}")
print(f"Formula              : {atoms.get_chemical_formula()}")
print()

Atoms per frame      : 6
Electronic states    : 3
Number of geometries : 3731
Formula              : C2H4



## 2. Train X-MACE (AutoencoderExcitedMACE)

In [2]:
model = "AutoencoderExcitedMACE"
r_max = 5.0
max_num_epochs = 10
lr = 0.0001
energy_weight = 100.0
forces_weight = 100.0
device = "cpu"

In [3]:
xmace_cmd = f"""
python scripts/run_train.py \
  --name="energies_forces" \
  --train_file="{train_file}" \
  --seed=100 \
  --valid_fraction=0.1 \
  --E0s='average' \
  --model="{model}" \
  --r_max={r_max} \
  --batch_size=10 \
  --n_energies={n_energies} \
  --correlation=3 \
  --max_num_epochs={max_num_epochs} \
  --ema \
  --lr={lr} \
  --ema_decay=0.99 \
  --default_dtype="float32" \
  --device={device} \
  --hidden_irreps="128x0e + 128x1o" \
  --MLP_irreps='128x0e' \
  --num_radial_basis=8 \
  --num_interactions=2 \
  --energy_weight={energy_weight} \
  --forces_weight={forces_weight} \
  --error_table="EnergyNacsDipoleMAE"
"""

import subprocess
subprocess.run(xmace_cmd, shell=True, check=True)

ERROR:root:No token file found. Also make sure that a [prod] section with a 'token = value' assignment exists.
INFO:root:===========VERIFYING SETTINGS===========
INFO:root:MACE version: 0.3.6
DEBUG:root:Configuration: Namespace(name='energies_forces', seed=100, work_dir='.', nacs_key='smooth_nacs', log_dir='./logs', model_dir='.', checkpoints_dir='./checkpoints', results_dir='./results', downloads_dir='./downloads', device='cpu', default_dtype='float32', distributed=False, log_level='INFO', n_energies=3, error_table='EnergyNacsDipoleMAE', model='AutoencoderExcitedMACE', r_max=5.0, num_permutational_invariant=16, radial_type='bessel', num_radial_basis=8, num_cutoff_basis=5, pair_repulsion=False, distance_transform='None', interaction='RealAgnosticResidualInteractionBlock', interaction_first='RealAgnosticResidualInteractionBlock', max_ell=3, correlation=3, num_interactions=2, MLP_irreps='128x0e', radial_MLP='[64, 64, 64]', hidden_irreps='128x0e + 128x1o', num_channels=128, max_L=1, gate=

2026-07-02 13:39:22.946 INFO: ===========VERIFYING SETTINGS===========
2026-07-02 13:39:22.946 INFO: MACE version: 0.3.6
2026-07-02 13:39:22.946 INFO: Using CPU
2026-07-02 13:39:22.980 INFO: 
2026-07-02 13:39:22.980 INFO: ===========LOADING INPUT DATA===========


INFO:root:Using random 10% of training set for validation with indices saved in: ./valid_indices_100.txt
INFO:root:Atomic Numbers used: [np.int64(1), np.int64(6)]
INFO:root:Isolated Atomic Energies (E0s) not in training file, using command line argument
INFO:root:Computing average Atomic Energies using least squares regression
INFO:root:Atomic Energies used (z: eV): {1: -423.21686569816916, 6: -211.60843284908267}


2026-07-02 13:39:24.255 INFO: Using random 10% of training set for validation with indices saved in: ./valid_indices_100.txt
2026-07-02 13:39:24.260 INFO: Atomic Numbers used: [np.int64(1), np.int64(6)]
2026-07-02 13:39:24.260 INFO: Isolated Atomic Energies (E0s) not in training file, using command line argument
2026-07-02 13:39:24.260 INFO: Computing average Atomic Energies using least squares regression
2026-07-02 13:39:24.281 INFO: Atomic Energies used (z: eV): {1: -423.21686569816916, 6: -211.60843284908267}


INFO:root:
INFO:root:===========MODEL DETAILS===========
/home/yutong/micromamba/envs/x-mace-env/lib/python3.13/site-packages/torch/cuda/__init__.py:174: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0
/home/yutong/micromamba/envs/x-mace-env/lib/python3.13/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


...

INFO:root:Evaluating valid ...


2026-07-02 14:03:27.656 INFO: Evaluating valid ...
2026-07-02 14:03:33.273 INFO: Error-table on TRAIN and VALID:
+-------------+----------+----------+----------+----------+
| config_type |  MAE E   |  MAE F   | MAE nacs | MAE socs |
+-------------+----------+----------+----------+----------+
|    train    |   1454.4 |   1230.8 |      N/A |      N/A |
|    valid    |   1429.9 |   1217.1 |      N/A |      N/A |
+-------------+----------+----------+----------+----------+
2026-07-02 14:03:33.274 INFO: Saving model to checkpoints/energies_forces_run-100.model
2026-07-02 14:03:33.352 INFO: Compiling model, saving metadata to energies_forces_compiled.model


INFO:root:Error-table on TRAIN and VALID:
+-------------+----------+----------+----------+----------+
| config_type |  MAE E   |  MAE F   | MAE nacs | MAE socs |
+-------------+----------+----------+----------+----------+
|    train    |   1454.4 |   1230.8 |      N/A |      N/A |
|    valid    |   1429.9 |   1217.1 |      N/A |      N/A |
+-------------+----------+----------+----------+----------+
INFO:root:Saving model to checkpoints/energies_forces_run-100.model
INFO:root:Compiling model, saving metadata to energies_forces_compiled.model
INFO:root:Done


2026-07-02 14:03:33.914 INFO: Done


CompletedProcess(args='\npython scripts/run_train.py   --name="energies_forces"   --train_file="data/A01_ethene_grid_static_CASSCF.xyz"   --seed=100   --valid_fraction=0.1   --E0s=\'average\'   --model="AutoencoderExcitedMACE"   --r_max=5.0   --batch_size=10   --n_energies=3   --correlation=3   --max_num_epochs=10   --ema   --lr=0.0001   --ema_decay=0.99   --default_dtype="float32"   --device=cpu   --hidden_irreps="128x0e + 128x1o"   --MLP_irreps=\'128x0e\'   --num_radial_basis=8   --num_interactions=2   --energy_weight=100.0   --forces_weight=100.0   --error_table="EnergyNacsDipoleMAE"\n', returncode=0)

## 3. Calculator

In [20]:
import sys
sys.path.insert(0, ".")   # adjust if mace.py lives elsewhere
from mace.calculators import MACECalculator

calculate_file = "data/A01_ethene_grid_static_CASSCF.xyz"
calculate_index = 0
model_path = "energies_forces.model"

atoms = ase.io.read(calculate_file, index=calculate_index)
print(f"Number of atoms  : {len(atoms)}")
print(f"Chemical formula : {atoms.get_chemical_formula()}")
print(f"Positions shape  : {atoms.positions.shape}")
print(f"Available info keys: {list(atoms.info.keys())}")

calc = MACECalculator(
    model_paths=model_path,
    device=device,
    default_dtype="float32",   # match training dtype
    model_type="MACE",
)

print(f"\nCalculator loaded successfully.")

Number of atoms  : 6
Chemical formula : C2H4
Positions shape  : (6, 3)
Available info keys: ['REF_energy', 'REF_forces']

Calculator loaded successfully.


In [16]:
# Attach the calculator
atoms.calc = calc

# Trigger calculation — ASE calls calc.calculate() internally
predicted_energies = atoms.get_potential_energy()   # returns array (n_states,)
predicted_forces   = calc.results["forces"]          # shape (N_atoms, n_states, 3)

print("=== Energies ===")
print(f"Shape  : {np.array(predicted_energies).shape}")
for i, e in enumerate(np.array(predicted_energies).flatten()):
    print(f"  State S{i}: {e:.6f} eV")

print()
print("=== Forces ===")
print(f"Shape  : {predicted_forces.shape}   (N_atoms, n_states, 3)")
print(f"Max |F| across all states: {np.abs(predicted_forces).max():.4f} eV/Å")
for s in range(n_energies):
    f_rms = np.sqrt(np.mean(predicted_forces[:, s, :] ** 2))
    print(f"  State S{s} RMS force: {f_rms:.4f} eV/Å")

=== Energies ===
Shape  : (1, 3)
  State S0: -2121.030518 eV
  State S1: -2113.409180 eV
  State S2: -2109.041504 eV

=== Forces ===
Shape  : (6, 3, 3)   (N_atoms, n_states, 3)
Max |F| across all states: 34.2164 eV/Å
  State S0 RMS force: 8.7773 eV/Å
  State S1 RMS force: 8.4899 eV/Å
  State S2 RMS force: 11.4934 eV/Å
